# STEP 1 — PROJECT OVERVIEW (Markdown cell)
"""
This script processes EPC energy simulation outputs for multiple buildings and converts them into carbon emissions.

For each building folder:
- Reads `energy_results_tot.csv`
- Reads the global carbon intensity lookup (`carbon_kg_per_Wh.csv`)
- Multiplies each energy value by the corresponding carbon factor (matched by fuel type and time step)
- Multiplies the result by 0.5
- Writes the result into `carbon_results.csv` inside the same TOTAL folder

Key assumptions:
- All time steps align between energy and carbon lookup tables
- Column naming follows the provided mapping rules
- Each building folder is named using BUILDING_REFERENCE_NUMBER
"""

In [1]:
# STEP 2 — IMPORTS AND FILE PATHS

import os
import pandas as pd
from pathlib import Path

# Root directories
EPC_FILE = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_hp_upgrade.csv"
BASELINE_DIR = "/Users/jakegehrung/Desktop/data_raw/baseline_models"
CARBON_LOOKUP_FILE = "/Users/jakegehrung/Desktop/data_raw/REFERENCE FILES/carbon_kg_per_Wh.csv"

In [2]:
# STEP 3 — LOAD CARBON INTENSITY LOOKUP TABLE

carbon_df = pd.read_csv(CARBON_LOOKUP_FILE)

# Ensure Time is usable for alignment
carbon_df["Time"] = carbon_df["Time"].astype(str)

# Set Time as index for fast lookup
carbon_df = carbon_df.set_index("Time")

# Preview check (optional)
carbon_df.head()

,electricity,lpg,mineral_wood,mains_gas,oil,wood_logs,wood_pellets,wood_chips,coal
Time,,,,,,,,,
00:30:00,0.000136,0.000241,0.000087,0.00021,0.000298,0.000028,0.000053,0.000023,0.000395
01:00:00,0.000136,0.000241,0.000087,0.00021,0.000298,0.000028,0.000053,0.000023,0.000395
01:30:00,0.000136,0.000241,0.000087,0.00021,0.000298,0.000028,0.000053,0.000023,0.000395
02:00:00,0.000136,0.000241,0.000087,0.00021,0.000298,0.000028,0.000053,0.000023,0.000395
02:30:00,0.000136,0.000241,0.000087,0.00021,0.000298,0.000028,0.000053,0.000023,0.000395


In [3]:
# STEP 4 — DEFINE COLUMN MAPPING AND PROCESSING FUNCTION

# Map EPC energy columns -> carbon intensity columns
COLUMN_MAP = {
    "electricity_exp_1": "electricity",
    "electricity_tot_1": "electricity",
    "lpg_tot_1": "lpg",
    "mineral_wood_tot_1": "mineral_wood",
    "mains_gas_tot_1": "mains_gas",
    "oil_tot_1": "oil",
    "wood_logs_tot_1": "wood_logs",
    "wood_pellets_tot_1": "wood_pellets",
    "wood_chips_tot_1": "wood_chips",
    "coal_tot_1": "coal",

    "electricity_exp_2": "electricity",
    "electricity_tot_2": "electricity",
    "lpg_tot_2": "lpg",
    "mineral_wood_tot_2": "mineral_wood",
    "mains_gas_tot_2": "mains_gas",
    "oil_tot_2": "oil",
    "wood_logs_tot_2": "wood_logs",
    "wood_pellets_tot_2": "wood_pellets",
    "wood_chips_tot_2": "wood_chips",
    "coal_tot_2": "coal",

    "electricity_exp_3": "electricity",
    "electricity_tot_3": "electricity",
    "lpg_tot_3": "lpg",
    "mineral_wood_tot_3": "mineral_wood",
    "mains_gas_tot_3": "mains_gas",
    "oil_tot_3": "oil",
    "wood_logs_tot_3": "wood_logs",
    "wood_pellets_tot_3": "wood_pellets",
    "wood_chips_tot_3": "wood_chips",
    "coal_tot_3": "coal",

    "electricity_exp_4": "electricity",
    "electricity_tot_4": "electricity",
    "lpg_tot_4": "lpg",
    "mineral_wood_tot_4": "mineral_wood",
    "mains_gas_tot_4": "mains_gas",
    "oil_tot_4": "oil",
    "wood_logs_tot_4": "wood_logs",
    "wood_pellets_tot_4": "wood_pellets",
    "wood_chips_tot_4": "wood_chips",
    "coal_tot_4": "coal",

    "electricity_exp_5": "electricity",
    "electricity_tot_5": "electricity",
    "lpg_tot_5": "lpg",
    "mineral_wood_tot_5": "mineral_wood",
    "mains_gas_tot_5": "mains_gas",
    "oil_tot_5": "oil",
    "wood_logs_tot_5": "wood_logs",
    "wood_pellets_tot_5": "wood_pellets",
    "wood_chips_tot_5": "wood_chips",
    "coal_tot_5": "coal",

    "electricity_exp_6": "electricity",
    "electricity_tot_6": "electricity",
    "lpg_tot_6": "lpg",
    "mineral_wood_tot_6": "mineral_wood",
    "mains_gas_tot_6": "mains_gas",
    "oil_tot_6": "oil",
    "wood_logs_tot_6": "wood_logs",
    "wood_pellets_tot_6": "wood_pellets",
    "wood_chips_tot_6": "wood_chips",
    "coal_tot_6": "coal",

    "electricity_exp_7": "electricity",
    "electricity_tot_7": "electricity",
    "lpg_tot_7": "lpg",
    "mineral_wood_tot_7": "mineral_wood",
    "mains_gas_tot_7": "mains_gas",
    "oil_tot_7": "oil",
    "wood_logs_tot_7": "wood_logs",
    "wood_pellets_tot_7": "wood_pellets",
    "wood_chips_tot_7": "wood_chips",
    "coal_tot_7": "coal",

    "electricity_exp_8": "electricity",
    "electricity_tot_8": "electricity",
    "lpg_tot_8": "lpg",
    "mineral_wood_tot_8": "mineral_wood",
    "mains_gas_tot_8": "mains_gas",
    "oil_tot_8": "oil",
    "wood_logs_tot_8": "wood_logs",
    "wood_pellets_tot_8": "wood_pellets",
    "wood_chips_tot_8": "wood_chips",
    "coal_tot_8": "coal",

    "electricity_exp_9": "electricity",
    "electricity_tot_9": "electricity",
    "lpg_tot_9": "lpg",
    "mineral_wood_tot_9": "mineral_wood",
    "mains_gas_tot_9": "mains_gas",
    "oil_tot_9": "oil",
    "wood_logs_tot_9": "wood_logs",
    "wood_pellets_tot_9": "wood_pellets",
    "wood_chips_tot_9": "wood_chips",
    "coal_tot_9": "coal"    
}


def process_building(building_id: str):
    """
    Process a single building folder and generate carbon_results.csv
    """
    total_path = os.path.join(BASELINE_DIR, building_id, "TOTAL")
    
    energy_file = os.path.join(total_path, "energy_results_tot.csv")
    output_file = os.path.join(total_path, "carbon_results.csv")
    
    if not os.path.exists(energy_file):
        print(f"[SKIP] Missing: {energy_file}")
        return
    
    energy_df = pd.read_csv(energy_file)
    energy_df["Time"] = energy_df["Time"].astype(str)
    energy_df = energy_df.set_index("Time")
    
    # Prepare output dataframe
    carbon_out = pd.DataFrame(index=energy_df.index)
    
    # Iterate through energy columns
    for col in energy_df.columns:
        if col not in COLUMN_MAP:
            continue
        
        fuel = COLUMN_MAP[col]
        
        if fuel not in carbon_df.columns:
            print(f"[WARNING] Missing carbon factor for fuel: {fuel}")
            continue
        
        # Align by Time index
        carbon_factor = carbon_df[fuel].reindex(energy_df.index)
        
        carbon_out[col] = energy_df[col] * carbon_factor * 0.5
    
    # Save result
    carbon_out.reset_index().to_csv(output_file, index=False)
    print(f"[DONE] {building_id}")

In [4]:
# STEP 5 (MANUAL TEST) — SPECIFIC BUILDING

#test_building_id = "1001829067"  # replace if needed

#print("Testing with building:", test_building_id)

#process_building(test_building_id)

In [5]:

# STEP 5 — RUN FOR ALL BUILDINGS

epc_df = pd.read_csv(EPC_FILE)

building_ids = epc_df["BUILDING_REFERENCE_NUMBER"].dropna().astype(str).unique()

print(f"Found {len(building_ids)} buildings")

for b_id in building_ids:
    process_building(b_id)

print("All processing complete.")

Found 117 buildings
[DONE] 1001829067
[DONE] 1001951858
[DONE] 1001829071
[DONE] 1234761001
[DONE] 1001991633
[DONE] 1001664929
[DONE] 1001829059
[DONE] 1001063639
[DONE] 1234761000
[DONE] 1236594950
[DONE] 1001664925
[DONE] 1001906271
[DONE] 1000238911
[DONE] 1001829057
[DONE] 1234760999
[DONE] 1002143357
[DONE] 1001951854
[DONE] 1001829069
[DONE] 1002313096
[DONE] 1002143351
[DONE] 1001870854
[DONE] 1001870864
[DONE] 1002143293
[DONE] 1002143352
[DONE] 1002143353
[DONE] 1234647955
[DONE] 1002313093
[DONE] 1001991629
[DONE] 1001991628
[DONE] 1000238920
[DONE] 1001829085
[DONE] 1001063646
[DONE] 1001829058
[DONE] 1234806523
[DONE] 1001664941
[DONE] 1236034494
[DONE] 1000336709
[DONE] 1234761002
[DONE] 1002143355
[DONE] 1001906269
[DONE] 1001870855
[DONE] 1001664944
[DONE] 1000336711
[DONE] 1001829079
[DONE] 1001870859
[DONE] 1001664928
[DONE] 1234806526
[DONE] 1001951889
[DONE] 1001627558
[DONE] 1235812262
[DONE] 1001627567
[DONE] 1001627542
[DONE] 1001627549
[DONE] 1002143348
[DONE] 1